In [ ]:
# Fix dependency conflicts
!pip install --upgrade --force-reinstall "transformers" "tokenizers" "huggingface-hub<1.0"

In [ ]:
!pip install "huggingface-hub<1.0" --upgrade
%pip install -Uq "openvino>=2024.5.0" "openvino-tokenizers>=2024.5.0" "openvino-genai>=2024.5.0" huggingface_hub

In [25]:
def calculate_and_print_metrics(result, start_time, end_time, pipe, label="Results"):
    """
    Calculate and print generation metrics including tokens per second.
    
    Args:
        result: The generation result object
        start_time: Start time of generation
        end_time: End time of generation
        pipe: The LLM pipeline object
        label: Label for the results (e.g., "Original Speculation", "Dynamic Spec")
    """
    # Calculate tokens per second
    generation_time = end_time - start_time
    
    # Get the actual generated text
    generated_text = result.texts[0] if hasattr(result, 'texts') else str(result)
    
    # Get the tokenizer from the pipeline to count actual tokens
    tokenizer = pipe.get_tokenizer()
    encoded = tokenizer.encode(generated_text)
    
    # Get the input_ids from the TokenizedInputs object
    num_tokens = encoded.input_ids.shape[1] if hasattr(encoded, 'input_ids') else len(encoded.input_ids)
    
    tokens_per_second = num_tokens / generation_time
    
    # Print results
    print(f"\n{label}: ")
    print(f"Generation time: {generation_time:.2f}s")
    print(f"Generated tokens: {num_tokens}")
    print(f"Tokens per second: {tokens_per_second:.2f}")

In [23]:
from pathlib import Path
import huggingface_hub as hf_hub

draft_model_id = "OpenVINO/Phi-3-mini-FastDraft-50M-int8-ov"
target_model_id = "OpenVINO/Phi-3-mini-4k-instruct-int4-ov"

draft_model_path = Path(draft_model_id.split("/")[-1])
target_model_path = Path(target_model_id.split("/")[-1])

if not draft_model_path.exists():
    hf_hub.snapshot_download(draft_model_id, local_dir=draft_model_path)
if not target_model_path.exists():
    hf_hub.snapshot_download(target_model_id, local_dir=target_model_path)

In [24]:
import requests

r = requests.get(
    url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
)
open("notebook_utils.py", "w").write(r.text)

from notebook_utils import device_widget

device = device_widget(default="CPU", exclude=["NPU", "AUTO"])

device

Dropdown(description='Device:', options=('CPU',), value='CPU')

*** No Speculation ***

In [26]:
import openvino_genai as ov_genai
import time

pipe = ov_genai.LLMPipeline(target_model_path, device.value)

config = ov_genai.GenerationConfig()
config.max_new_tokens = 100

def streamer(subword):
    print(subword, end="", flush=True)
    # Return flag corresponds whether generation should be stopped.
    # False means continue generation.
    return False

start_time = time.perf_counter()
result = pipe.generate(["Sun is yellow because"], config, streamer=streamer)
end_time = time.perf_counter()

Sunlight appears yellow because of the way Earth's atmosphere scatters sunlight. Sunlight is actually a mix of all colors, which combine to form white light. When sunlight passes through the Earth's atmosphere, shorter wavelengths of light (blue and violet) are scattered in all directions by the gases and particles in the air. This scattering causes the sky to look blue. However, the longer wavelengths of light (red, orange, and

In [27]:
calculate_and_print_metrics(result, start_time, end_time, pipe, label="No Speculation Results")


No Speculation Results: 
Generation time: 4.71s
Generated tokens: 100
Tokens per second: 21.25


In [28]:
import gc
del pipe
gc.collect()

0

In [29]:
scheduler_config = ov_genai.SchedulerConfig()
# cache params
scheduler_config.cache_size = 0
scheduler_config.num_kv_blocks = 2048 // 8
scheduler_config.max_num_batched_tokens = 2048

draft_model = ov_genai.draft_model(draft_model_path, device.value)

pipe = ov_genai.LLMPipeline(target_model_path, device.value, draft_model=draft_model, scheduler_config=scheduler_config)

config = ov_genai.GenerationConfig()
config.max_new_tokens = 100
config.num_assistant_tokens = 5
start_time = time.perf_counter()
result = pipe.generate(["Sun is yellow because"], config, streamer=streamer)
end_time = time.perf_counter()

Sunlight appears yellow because of the way Earth's atmosphere scatters sunlight. Sunlight is actually a mix of all colors, which combine to form white light. When sunlight passes through the Earth's atmosphere, shorter wavelengths (blue and violet) are scattered in all directions by the gases and particles in the air. This scattering causes the sky to look blue. However, the longer wavelengths (red, orange, and yellow) are less

In [30]:
calculate_and_print_metrics(result, start_time, end_time, pipe, label="Original Speculation Results")


Original Speculation Results: 
Generation time: 3.71s
Generated tokens: 100
Tokens per second: 26.98


*** This is the Dynamic Spec Code ***

In [33]:
config = ov_genai.GenerationConfig()
config.max_new_tokens = 100
config.assistant_confidence_threshold = 0.1
#config.num_assistant_tokens = 5
start_time = time.perf_counter()
result = pipe.generate(["Sun is yellow because"], config, streamer)
end_time = time.perf_counter()

Sun

light appears yellow because of the way Earth's atmosphere scatters sunlight. Sunlight is actually a mix of all colors, which combine to form white light. When sunlight passes through the Earth's atmosphere, shorter wavelengths (blue and violet) are scattered in all directions by the gases and particles in the air. This scattering causes the sky to look blue. However, the longer wavelengths (red, orange, and yellow) are less

In [34]:
calculate_and_print_metrics(result, start_time, end_time, pipe, label="Dynamic Spec Results")


Dynamic Spec Results: 
Generation time: 2.97s
Generated tokens: 100
Tokens per second: 33.65


## Cold Start Mitigation Strategies

The following cell demonstrates techniques to minimize cold start effects in OpenVINO pipelines:

In [ ]:
import os

def warm_up_pipeline(pipe, num_warmup_runs=3):
    """
    Warm up the pipeline to minimize cold start effects.
    
    Args:
        pipe: The OpenVINO LLM pipeline
        num_warmup_runs: Number of warm-up iterations
    """
    print("🔥 Warming up pipeline...")
    
    # Simple warm-up configuration
    warmup_config = ov_genai.GenerationConfig()
    warmup_config.max_new_tokens = 5  # Short generation for warm-up
    
    # Silent streamer for warm-up (no output)
    def silent_streamer(subword):
        return False
    
    for i in range(num_warmup_runs):
        print(f"  Warm-up run {i+1}/{num_warmup_runs}")
        pipe.generate(["Hello"], warmup_config, streamer=silent_streamer)
    
    print("✅ Pipeline warmed up!")

# Enable OpenVINO model caching (reduces compilation time on subsequent runs)
os.environ["OPENVINO_CACHE_DIR"] = "./ov_cache"
print(f"OpenVINO cache directory set to: {os.environ.get('OPENVINO_CACHE_DIR')}")

# Demonstrate warm-up
warm_up_pipeline(pipe)

In [ ]:
# Now test the warmed-up pipeline performance
print("\n" + "="*50)
print("🚀 Testing performance after warm-up:")
print("="*50)

config = ov_genai.GenerationConfig()
config.max_new_tokens = 100
config.assistant_confidence_threshold = 0.1

start_time = time.perf_counter()
result = pipe.generate(["Sun is yellow because"], config, streamer)
end_time = time.perf_counter()

calculate_and_print_metrics(result, start_time, end_time, pipe, label="Warmed-up Dynamic Spec Results")